In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from glide.common_components.constants import NPIX, MASK_L1A_FOV_R
import glide.science_data_processing.L1A as L1A 
from glide.common_components.utils import mask_average, circular_mask

# --- Configuration ---
BASE_PATH = "/home/jacob/products/L1A/"
IMAGER = "WFI"
FILENAMES = [f"CARRUTHERS_GCI-WFI_L1A-DRK_202510{day:02d}_v1.0.nc" for day in range(4, 16)]

def process_l1a_files(filenames, imager="WFI"):
    """Processes multiple files and returns concatenated time and average arrays."""
    all_avgs = []
    all_times = []
    
    # Pre-calculate mask once to save CPU
    mask = circular_mask(NPIX[imager], MASK_L1A_FOV_R[imager])

    for filename in filenames:
        try:
            with xr.open_dataset(BASE_PATH + filename, engine='netcdf4') as ds:
                l1a = L1A.L1A(ds)
                
                # Calculate mean signal
                # Note: ensure l1a.t_int is the correct parameter for your mask_average
                mean_val, _ = mask_average(l1a.images, mask, l1a.t_int)
                
                all_avgs.append(np.atleast_1d(mean_val))
                all_times.append(np.atleast_1d(l1a.time))
        except FileNotFoundError:
            print(f"Warning: {filename} not found. Skipping...")

    # Concatenate once at the end (much faster than repeated concatenation)
    return (np.concatenate(all_times).astype('datetime64[ns]'), 
            np.concatenate(all_avgs).astype('float64'))

def plot_L1A(time, avg, save_path=None):
    """Handles the visualization logic."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(time, avg, label='L1A Mean Signal', color='tab:blue', linewidth=1)
    
    ax.set(xlabel='Time', ylabel='Signal (units)', title='L1A Signal Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Improve date formatting on x-axis
    fig.autofmt_xdate()

    if save_path:
        plt.savefig(save_path)
        print(f"Plot saved to {save_path}")
    else:
        plt.show()

# --- Execution ---
if __name__ == "__main__":
    times, avgs = process_l1a_files(FILENAMES, IMAGER)
    
    if len(times) > 0:
        plot_L1A(times, avgs)
    else:
        print("No data found to plot.")


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.